In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,roc_auc_score,precision_score,recall_score,f1_score,matthews_corrcoef
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier


Importing the Dataset

In [2]:
df = pd.read_csv("Data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
print(df.head())
print(df.dtypes)

(7043, 21)
   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Co

In [3]:
missing_values = df.isnull().sum()
print(missing_values)

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [4]:
# Converting TOtal charges blank values to a number i.e. 0

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors="coerce")
print(df["TotalCharges"].isna().sum())
print(df[df["TotalCharges"].isna()]["tenure"].unique())
df["TotalCharges"] = df["TotalCharges"].fillna(0)
df = df.drop(columns=["customerID"])

print(df.shape)
print(df["TotalCharges"])

11
[0]
(7043, 20)
0         29.85
1       1889.50
2        108.15
3       1840.75
4        151.65
         ...   
7038    1990.50
7039    7362.90
7040     346.45
7041     306.60
7042    6844.50
Name: TotalCharges, Length: 7043, dtype: float64


In [5]:
# Encoding the target and categorical deatures
df["Churn"] = df["Churn"].map({"Yes" : 1, "No": 0})

# Separating features and Target
X = df.drop(columns=["Churn"])
Y = df["Churn"]

# check which columns are categorical (text) vs numeric
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numeric_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical columns :",categorical_cols)
print("Numerical columns : ",numeric_cols)

# One hot encode the categorical columns
X_encoded = pd.get_dummies(X,columns=categorical_cols,drop_first=True)

print(X_encoded.shape)
print(X_encoded.columns.tolist())

Categorical columns : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Numerical columns :  ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
(7043, 30)
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBill

/var/folders/5v/00dt4vxd00zf63gmq8rw457w0000gn/T/ipykernel_40110/1552018380.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include="object").columns.tolist()


In [6]:
# Splitting the dataset into Train and Test Dataset

X_train,X_test,y_train,y_test = train_test_split(
    X_encoded,Y,test_size=0.2,random_state=42,stratify= Y
)

# Scaling the original numeric columns
scaler = StandardScaler()
numeric_cols_to_scale = ["SeniorCitizen","tenure","MonthlyCharges","TotalCharges"]

X_train[numeric_cols_to_scale] = scaler.fit_transform(X_train[numeric_cols_to_scale])
X_test[numeric_cols_to_scale] = scaler.fit_transform(X_test[numeric_cols_to_scale])

print(X_train.shape,X_test.shape)
print(y_train.value_counts(normalize = True))
print(y_test.value_counts(normalize = True))

(5634, 30) (1409, 30)
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


In [7]:
# Implementing the Logistic Regression Model

log_reg = LogisticRegression(max_iter=1000,random_state=42)
log_reg.fit(X_train,y_train)

y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:,1]

metrics_log_reg = {
    "Model":"LogisticRegression",
    "Accuracy":accuracy_score(y_test,y_pred),
    "AUC": roc_auc_score(y_test,y_proba),
    "Precision":precision_score(y_test,y_pred),
    "Recall":recall_score(y_test,y_pred),
    "F1":f1_score(y_test,y_pred),
    "MCC":matthews_corrcoef(y_test,y_pred)


}

print(metrics_log_reg)

{'Model': 'LogisticRegression', 'Accuracy': 0.8048261178140526, 'AUC': 0.842111653620605, 'Precision': 0.65814696485623, 'Recall': 0.5508021390374331, 'F1': 0.5997088791848617, 'MCC': 0.4752743817582408}


In [8]:
# Implementing all five models

models = {
    "Logistic Regression" : log_reg,
    "Decision Tree" : DecisionTreeClassifier(random_state=42),
    "kNN" : KNeighborsClassifier(),
    "Naive Bayes" : GaussianNB(),
    "Random Forest" : RandomForestClassifier(random_state=42)
}

results = []

for name , model in models.items():
    if name != "LogisticRegression":
        model.fit(X_train,y_train)


    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:,1]


    results.append({
        "Model" : name,
        "Accuracy" : accuracy_score(y_test,y_pred),
        "AUC" : roc_auc_score(y_test,y_proba),
        "Precision" : precision_score(y_test,y_pred),
        "Recall" : recall_score(y_test,y_pred),
        "F1" : f1_score(y_test,y_pred),
        "MCC" : matthews_corrcoef(y_test,y_pred)
    })

results_df = pd.DataFrame(results)
print(results_df)



                 Model  Accuracy       AUC  Precision    Recall        F1  \
0  Logistic Regression  0.804826  0.842112   0.658147  0.550802  0.599709   
1        Decision Tree  0.733144  0.656134   0.497297  0.491979  0.494624   
2                  kNN  0.762243  0.786378   0.552846  0.545455  0.549125   
3          Naive Bayes  0.659333  0.809473   0.429708  0.866310  0.574468   
4        Random Forest  0.784954  0.816513   0.621993  0.483957  0.544361   

        MCC  
0  0.475274  
1  0.313347  
2  0.387706  
3  0.399147  
4  0.411964  


In [9]:
# Saving the models and exporting test data

import joblib
import os

os.makedirs("model",exist_ok=True)
for name,model in models.items():
    filename = name.lower().replace(" ","_")
    joblib.dump(model,f"model/{filename}.pkl")


joblib.dump(scaler,"model/scaler.pkl")
joblib.dump(X_encoded.columns.tolist(),"model/model_columns.pkl")

test_data = X_test.copy()
test_data["Churn"] = y_test
test_data.to_csv("test_data.csv",index = False)

print("Saved models:",os.listdir("model"))
print("test_data.csv shape:",test_data.shape)



Saved models: ['scaler.pkl', 'decision_tree.pkl', 'knn.pkl', 'logistic_regression.pkl', 'naive_bayes.pkl', 'model_columns.pkl', 'random_forest.pkl']
test_data.csv shape: (1409, 31)
